# DeepFake Shield — Complete Colab Notebook

## CELL 1 — Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


---

# CELL 2 — Check GPU



In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("❌ GPU NOT AVAILABLE")
    print("Go to Runtime → Change runtime type → T4 GPU")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


### You want to see something like:

```text
CUDA available: True
GPU: Tesla T4
GPU memory: 15.0 GB
```

**If CUDA is `False`, stop here and enable GPU.**

---

In [ ]:
%cd /content

!git clone https://github.com/GokulM8/deep-fake-shield.git

%cd /content/deep-fake-shield

!git status
!ls

/content
Cloning into 'deep-fake-shield'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 216 (delta 38), reused 211 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (216/216), 1.23 MiB | 16.86 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/deep-fake-shield
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
 artifacts   datasets		    docs       ml	   reports
 backend    'DeepFake Shield.pdf'   frontend   README.md   scripts


# CELL 3 — Define all paths

In [ ]:
from pathlib import Path

# Repository cloned in Colab
REPO_ROOT = Path("/content/deep-fake-shield")

# Dataset permanently stored in Google Drive
DATA_ROOT = Path(
    "/content/drive/MyDrive/DeepFake-Shield (1)/datasets/raw"
)

# Existing split files
SPLITS_ROOT = REPO_ROOT / "artifacts" / "splits"

TRAIN_CSV = SPLITS_ROOT / "train.csv"
VAL_CSV = SPLITS_ROOT / "val.csv"
TEST_CSV = SPLITS_ROOT / "test.csv"

# Output directories
EXPERIMENTS_ROOT = REPO_ROOT / "artifacts" / "experiments"
EVALUATION_ROOT = REPO_ROOT / "artifacts" / "evaluation"

# Permanent model location in Drive
DRIVE_MODEL_ROOT = Path(
    "/content/drive/MyDrive/DeepFake-Shield/models"
)

DRIVE_MODEL_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Dataset:", DATA_ROOT)
print("Splits:", SPLITS_ROOT)

Repository: /content/deep-fake-shield
Dataset: /content/drive/MyDrive/DeepFake-Shield (1)/datasets/raw
Splits: /content/deep-fake-shield/artifacts/splits


---

# CELL 4 — Verify repository

In [ ]:
print("Repository exists:", REPO_ROOT.exists())

if REPO_ROOT.exists():
    print("\nRepository contents:\n")

    for item in sorted(REPO_ROOT.iterdir()):
        print(
            "📁" if item.is_dir() else "📄",
            item.name
        )

Repository exists: True

Repository contents:

📄 .env.example
📁 .git
📄 .gitignore
📄 DeepFake Shield.pdf
📄 README.md
📁 artifacts
📁 backend
📁 datasets
📁 docs
📁 frontend
📁 ml
📁 reports
📁 scripts


You should see your project files.

---

# CELL 5 — Verify training scripts

In [ ]:
SCRIPTS_ROOT = REPO_ROOT / "scripts"

required_scripts = [
    "inspect_dataset.py",
    "split_dataset.py",
    "train_baseline.py",
    "evaluate_baseline.py",
    "infer_video.py"
]

print("Checking scripts...\n")

for script in required_scripts:

    path = SCRIPTS_ROOT / script

    if path.exists():
        print("✅", script)
    else:
        print("❌", script)

Checking scripts...

✅ inspect_dataset.py
✅ split_dataset.py
✅ train_baseline.py
✅ evaluate_baseline.py
✅ infer_video.py


You already confirmed these exist, so this should be all green.

---

# CELL 6 — Verify dataset

In [ ]:
!ls -lah /content/drive/MyDrive

total 372M
drwx------ 8 root root 4.0K Sep 15 12:57  .
drwx------ 5 root root 4.0K Sep 15 12:57  ..
-rw------- 1 root root  60K Feb  8  2026  00017641453916470ada4f6b66654880.pdf
-rw------- 1 root root  182 Jul  2 12:13 '2026 Briefing: The Landscape of Free LLM APIs and Local Inference.gdoc'
-rw------- 1 root root 249K Feb 15  2026  22f29d23-312b-4ab3-a94c-a334c83bf888.pdf
-rw------- 1 root root 543K Oct 25  2023 'Aadhar (1).pdf'
-rw------- 1 root root 937K Jun 26  2024 'Aadhar Card .pdf'
-rw------- 1 root root 190K Jun 15  2021 'Aadhar Card.pdf'
-rw------- 1 root root 970K Jan 20  2024  Aadhar.pdf
-rw------- 1 root root  182 May 13 19:45 'Applied_Statistics_FAT_Checklist BY COLLEGE PREP.gsheet'
-rw------- 1 root root 2.4M Jun 21 19:49  C5EC1709-DA19-4682-BB30-3F4E3886346D.PNG
-rw------- 1 root root 1.3M Sep 18  2024 'CALCULATOR WARRANTY RECEPT .pdf'
-rw------- 1 root root  182 Mar  1  2026 'CARBON‑X (1).gslides'
-rw------- 1 root root  182 Feb 27  2026 'CARBON‑X (2).gslides'
-rw------

In [ ]:
!find "/content/drive/MyDrive/DeepFake-Shield" -maxdepth 4 -type d | head -50
!ls -lah "/content/drive/MyDrive/DeepFake-Shield"

/content/drive/MyDrive/DeepFake-Shield
/content/drive/MyDrive/DeepFake-Shield/datasets
/content/drive/MyDrive/DeepFake-Shield/datasets/raw
/content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-real
/content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-Youtube-fake
/content/drive/MyDrive/DeepFake-Shield/datasets/raw/YouTube-real
/content/drive/MyDrive/DeepFake-Shield/artifacts
/content/drive/MyDrive/DeepFake-Shield/artifacts/splits
/content/drive/MyDrive/DeepFake-Shield/artifacts/experiments
/content/drive/MyDrive/DeepFake-Shield/artifacts/evaluation
/content/drive/MyDrive/DeepFake-Shield/artifacts/models
/content/drive/MyDrive/DeepFake-Shield/models
total 20K
drwx------ 5 root root 4.0K Sep 15 12:57 .
drwx------ 8 root root 4.0K Sep 15 12:57 ..
drwx------ 6 root root 4.0K Sep 15 11:08 artifacts
drwx------ 3 root root 4.0K Sep 15 11:05 datasets
drwx------ 2 root root 4.0K Sep 15 12:57 models


In [ ]:
!find "/content/drive/MyDrive/DeepFake-Shield" -type d \( -name "Celeb-Youtube-fake" -o -name "Celeb-real" -o -name "YouTube-real" \) 2>/dev/null

/content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-real
/content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-Youtube-fake
/content/drive/MyDrive/DeepFake-Shield/datasets/raw/YouTube-real


In [ ]:
from pathlib import Path

# Repository
REPO_ROOT = Path("/content/deep-fake-shield")

# Dataset
DATA_ROOT = Path("/content/drive/MyDrive/DeepFake-Shield/datasets/raw")

# Existing splits
SPLITS_ROOT = REPO_ROOT / "artifacts/splits"

TRAIN_CSV = SPLITS_ROOT / "train.csv"
VAL_CSV   = SPLITS_ROOT / "val.csv"
TEST_CSV  = SPLITS_ROOT / "test.csv"

print("Repo:", REPO_ROOT)
print("Dataset:", DATA_ROOT)
print("Splits:", SPLITS_ROOT)

print("\nDataset exists:", DATA_ROOT.exists())
print("Train CSV:", TRAIN_CSV.exists())
print("Val CSV:", VAL_CSV.exists())
print("Test CSV:", TEST_CSV.exists())

Repo: /content/deep-fake-shield
Dataset: /content/drive/MyDrive/DeepFake-Shield/datasets/raw
Splits: /content/deep-fake-shield/artifacts/splits

Dataset exists: True
Train CSV: True
Val CSV: True
Test CSV: True


In [ ]:
print("Dataset exists:", DATA_ROOT.exists())

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_ROOT}"
    )

print("\nDataset folders:\n")

for folder in sorted(DATA_ROOT.iterdir()):

    if folder.is_dir():

        count = len(list(folder.rglob("*.mp4")))

        print(
            f"{folder.name}: {count}"
        )

Dataset exists: True

Dataset folders:

Celeb-Youtube-fake: 5299
Celeb-real: 482
YouTube-real: 230


Expected:

```text
Celeb-real: 482
Celeb-Youtube-fake: 5299
YouTube-real: 230
```

---

# CELL 7 — Count all videos

In [ ]:
VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm"
}

videos = [
    p
    for p in DATA_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in VIDEO_EXTENSIONS
]

print("Total labeled videos available:", len(videos))

Total labeled videos available: 6011


Expected:

```text
Total labeled videos available: 6011
```

---

# CELL 8 — Check the split files

In [ ]:
print("Train CSV:", TRAIN_CSV.exists())
print("Validation CSV:", VAL_CSV.exists())
print("Test CSV:", TEST_CSV.exists())

Train CSV: True
Validation CSV: True
Test CSV: True


All three should be:

```text
True
```

---

# CELL 9 — Inspect split sizes

In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print(
    "\nTotal:",
    len(train_df) +
    len(val_df) +
    len(test_df)
)

Train: 4207
Validation: 902
Test: 902

Total: 6011


Expected:

```text
Train: 4207
Validation: 902
Test: 902

Total: 6011
```

**Do not recreate these splits.**

---

# CELL 10 — Inspect CSV structure

In [ ]:
print("Train columns:")
print(train_df.columns.tolist())

print("\nFirst 5 training rows:")
display(train_df.head())

Train columns:
['video_path', 'label', 'source', 'dataset', 'video_id', 'sha256']

First 5 training rows:


,video_path,label,source,dataset,video_id,sha256
0,/Users/gokulmallabathula/deep-fake/datasets/ra...,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:0,7062a11c8867b2bdef724a313412094511aa6b4ebe52a7...
1,/Users/gokulmallabathula/deep-fake/datasets/ra...,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:1,06d003c71bc0b79702298416e55391f38495f45346e7d5...
2,/Users/gokulmallabathula/deep-fake/datasets/ra...,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:10,9e66998c0e0645a97ad112013eae9ae62244f7bbac4467...
3,/Users/gokulmallabathula/deep-fake/datasets/ra...,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:1001,13964c2f76c78acda5769e7fae5ad4634d40d262fdec1a...
4,/Users/gokulmallabathula/deep-fake/datasets/ra...,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:1002,ba39d9b2c7581156222c350ea24c2ed03fdd07bebc6cdd...


This is important because the model script needs to know where the video path and label are stored.

---

# CELL 11 — Inspect training script arguments

In [ ]:
%cd /content/deep-fake-shield

!python scripts/train_baseline.py --help

/content/deep-fake-shield
usage: train_baseline.py [-h] [--train-csv TRAIN_CSV] [--val-csv VAL_CSV]
                         [--video-root VIDEO_ROOT] [--epochs EPOCHS]
                         [--limit LIMIT] [--frames-per-video FRAMES_PER_VIDEO]
                         [--batch-size BATCH_SIZE] [--workers WORKERS]
                         [--output-dir OUTPUT_DIR] [--pretrained]
                         [--seed SEED]

Train a small, reproducible EfficientNet face/frame baseline.

options:
  -h, --help            show this help message and exit
  --train-csv TRAIN_CSV
  --val-csv VAL_CSV
  --video-root VIDEO_ROOT
                        Mounted dataset root containing the raw dataset
                        folders
  --epochs EPOCHS
  --limit LIMIT         Limit each split for a sanity run; 0 means all rows
  --frames-per-video FRAMES_PER_VIDEO
  --batch-size BATCH_SIZE
  --workers WORKERS
  --output-dir OUTPUT_DIR
  --pretrained
  --seed SEED


You should see the arguments you were using previously, including:

```text
--train-csv
--val-csv
--video-root
--pretrained
--epochs
--limit
--frames-per-video
--batch-size
--workers
--output-dir
```

---

# CELL 12 — Inspect evaluation arguments

In [ ]:
!python scripts/evaluate_baseline.py --help

usage: evaluate_baseline.py [-h] [--csv CSV] [--video-root VIDEO_ROOT]
                            --checkpoint CHECKPOINT [--output-dir OUTPUT_DIR]
                            [--frames FRAMES] [--limit LIMIT]

Evaluate a frame baseline at video level on a held-out split.

options:
  -h, --help            show this help message and exit
  --csv CSV
  --video-root VIDEO_ROOT
                        Mounted dataset root containing the raw dataset
                        folders
  --checkpoint CHECKPOINT
  --output-dir OUTPUT_DIR
  --frames FRAMES
  --limit LIMIT


---

# CELL 13 — Inspect inference arguments

In [ ]:
!python scripts/infer_video.py --help

usage: infer_video.py [-h] --video VIDEO --checkpoint CHECKPOINT
                      [--frames FRAMES] [--output OUTPUT]

Analyze one video with a trained EfficientNet baseline.

options:
  -h, --help            show this help message and exit
  --video VIDEO
  --checkpoint CHECKPOINT
  --frames FRAMES
  --output OUTPUT


---

# CELL 14 — Check class distribution

First let's determine the label column automatically.

In [ ]:
print("Train columns:", train_df.columns.tolist())

Train columns: ['video_path', 'label', 'source', 'dataset', 'video_id', 'sha256']


Then run:

In [ ]:
for col in train_df.columns:
    print(
        f"\nColumn: {col}"
    )

    try:
        print(
            train_df[col].value_counts().head(10)
        )
    except:
        pass


Column: video_path
video_path
/Users/gokulmallabathula/deep-fake/datasets/raw/YouTube-real/99.mp4            1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/0.mp4       1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1.mp4       1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/10.mp4      1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1001.mp4    1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1002.mp4    1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1003.mp4    1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1004.mp4    1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1006.mp4    1
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1007.mp4    1
Name: count, dtype: int64

Column: label
label
1    3709
0     498
Name: count, dtype: int64

Column: source
Series([], Name: count, dtype: int64)

Column: dat

We need to identify the column containing `real/fake` or `0/1`.

---

# CELL 15 — Verify paths inside the CSV

This is extremely important because the CSVs were originally created on your Mac.

In [ ]:
print(train_df.head(10).to_string(index=False))

                                                                 video_path  label  source            dataset                video_id                                                           sha256
   /Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/0.mp4      1     NaN Celeb-Youtube-fake    Celeb-Youtube-fake:0 7062a11c8867b2bdef724a313412094511aa6b4ebe52a792a9a07deb0ecec6cf
   /Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1.mp4      1     NaN Celeb-Youtube-fake    Celeb-Youtube-fake:1 06d003c71bc0b79702298416e55391f38495f45346e7d576435a5418d251da10
  /Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/10.mp4      1     NaN Celeb-Youtube-fake   Celeb-Youtube-fake:10 9e66998c0e0645a97ad112013eae9ae62244f7bbac4467d16bcf7747636e8e63
/Users/gokulmallabathula/deep-fake/datasets/raw/Celeb-Youtube-fake/1001.mp4      1     NaN Celeb-Youtube-fake Celeb-Youtube-fake:1001 13964c2f76c78acda5769e7fae5ad4634d40d262fdec1a8e3939dfe018189f65
/User

If your CSV contains something like:

```text
datasets/raw/Celeb-Youtube-fake/100.mp4
```

that's good.

If it contains:

```text
/Users/gokulmallabathula/deep-fake/datasets/raw/...
```

that's a problem and we'll convert it.

# Fix: Clean Mac paths in CSVs

In [ ]:
# Function to clean video paths
def clean_mac_path(path_series):
    # This assumes the unwanted part is always '/Users/gokulmallabathula/deep-fake/'
    # and the desired relative path starts after 'datasets/raw/'
    # The DATA_ROOT already includes 'datasets/raw'
    return path_series.apply(lambda p: p.replace('/Users/gokulmallabathula/deep-fake/', ''))

# Apply the cleaning function to all relevant DataFrames
train_df['video_path'] = clean_mac_path(train_df['video_path'])
val_df['video_path'] = clean_mac_path(val_df['video_path'])
test_df['video_path'] = clean_mac_path(test_df['video_path'])

print("Cleaned paths in train_df:")
display(train_df.head())

print("\nFirst 10 training rows after cleaning:")
print(train_df.head(10).to_string(index=False))

Cleaned paths in train_df:


,video_path,label,source,dataset,video_id,sha256
0,datasets/raw/Celeb-Youtube-fake/0.mp4,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:0,7062a11c8867b2bdef724a313412094511aa6b4ebe52a7...
1,datasets/raw/Celeb-Youtube-fake/1.mp4,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:1,06d003c71bc0b79702298416e55391f38495f45346e7d5...
2,datasets/raw/Celeb-Youtube-fake/10.mp4,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:10,9e66998c0e0645a97ad112013eae9ae62244f7bbac4467...
3,datasets/raw/Celeb-Youtube-fake/1001.mp4,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:1001,13964c2f76c78acda5769e7fae5ad4634d40d262fdec1a...
4,datasets/raw/Celeb-Youtube-fake/1002.mp4,1,NaN,Celeb-Youtube-fake,Celeb-Youtube-fake:1002,ba39d9b2c7581156222c350ea24c2ed03fdd07bebc6cdd...



First 10 training rows after cleaning:
                              video_path  label  source            dataset                video_id                                                           sha256
   datasets/raw/Celeb-Youtube-fake/0.mp4      1     NaN Celeb-Youtube-fake    Celeb-Youtube-fake:0 7062a11c8867b2bdef724a313412094511aa6b4ebe52a792a9a07deb0ecec6cf
   datasets/raw/Celeb-Youtube-fake/1.mp4      1     NaN Celeb-Youtube-fake    Celeb-Youtube-fake:1 06d003c71bc0b79702298416e55391f38495f45346e7d576435a5418d251da10
  datasets/raw/Celeb-Youtube-fake/10.mp4      1     NaN Celeb-Youtube-fake   Celeb-Youtube-fake:10 9e66998c0e0645a97ad112013eae9ae62244f7bbac4467d16bcf7747636e8e63
datasets/raw/Celeb-Youtube-fake/1001.mp4      1     NaN Celeb-Youtube-fake Celeb-Youtube-fake:1001 13964c2f76c78acda5769e7fae5ad4634d40d262fdec1a8e3939dfe018189f65
datasets/raw/Celeb-Youtube-fake/1002.mp4      1     NaN Celeb-Youtube-fake Celeb-Youtube-fake:1002 ba39d9b2c7581156222c350ea24c2ed03fdd07beb

---

# CELL 16 — Test one CSV video against Drive



In [ ]:
from pathlib import Path

# Show every value from the first row
first_row = train_df.iloc[0]

print(first_row)

video_path                datasets/raw/Celeb-Youtube-fake/0.mp4
label                                                         1
source                                                      NaN
dataset                                      Celeb-Youtube-fake
video_id                                   Celeb-Youtube-fake:0
sha256        7062a11c8867b2bdef724a313412094511aa6b4ebe52a7...
Name: 0, dtype: object


In [ ]:
for col in train_df.columns:
    print(col, "=", first_row[col])

video_path = datasets/raw/Celeb-Youtube-fake/0.mp4
label = 1
source = nan
dataset = Celeb-Youtube-fake
video_id = Celeb-Youtube-fake:0
sha256 = 7062a11c8867b2bdef724a313412094511aa6b4ebe52a792a9a07deb0ecec6cf


This tells us the exact video path format.

---

# CELL 17 — Make sure we are in the repository



In [ ]:
%cd /content/deep-fake-shield

print("Current directory:")
!pwd

/content/deep-fake-shield
Current directory:
/content/deep-fake-shield


Expected:

```text
/content/deep-fake-shield
```

---

# CELL 18 — Run the existing dataset audit

**Do this for project verification only.**



In [ ]:
!python scripts/inspect_dataset.py

{
  "dataset_root": "/content/deep-fake-shield/datasets/raw",
  "total_videos": 0,
  "readable_videos": 0,
  "corrupted_or_unreadable_videos": 0,
  "class_distribution": {},
  "videos_by_dataset": {},
  "videos_by_extension": {},
  "duration_seconds": {
    "average": null,
    "minimum": null,
    "maximum": null
  },
  "average_fps": null,
  "resolution_distribution": {},
  "source_grouping": {
    "available": false,
    "reason": "The dataset is flat and contains no identity/source metadata or source subdirectories."
  }
}
Duplicate filenames: 0
Duplicate hashes: 0
Report: /content/deep-fake-shield/artifacts/dataset_report.json


If it reports:

```text
total_videos: 0
```

**that's expected with your current script**, because that script defaults to:

```text
/content/deep-fake-shield/datasets/raw
```

while your real dataset is in Drive.

Don't let that stop us. Our training command will explicitly provide the correct dataset path.

---

# CELL 19 — Create experiment directory



In [ ]:
SUBMISSION_DIR = (
    REPO_ROOT
    / "artifacts"
    / "experiments"
    / "submission_v1"
)

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(SUBMISSION_DIR)

/content/deep-fake-shield/artifacts/experiments/submission_v1


---

# CELL 20 — GPU sanity test



In [ ]:
import torch
import time

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":

    x = torch.randn(
        32,
        3,
        224,
        224,
        device=device
    )

    start = time.time()

    for _ in range(100):
        y = x * 2.0

    torch.cuda.synchronize()

    elapsed = time.time() - start

    print(
        "GPU test completed in:",
        round(elapsed, 3),
        "seconds"
    )

Device: cuda
GPU test completed in: 0.052 seconds


---

# 🚨 CELL 21 — 100-video sanity training

**Do this before full training.**



In [ ]:
import cv2

print("OpenCV version:", getattr(cv2, "__version__", "unknown"))
print("cv2 location:", getattr(cv2, "__file__", "unknown"))
print("Has CascadeClassifier:", hasattr(cv2, "CascadeClassifier"))

OpenCV version: 5.0.0
cv2 location: /usr/local/lib/python3.13/dist-packages/cv2/__init__.py
Has CascadeClassifier: False


In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
Found existing installation: opencv-contrib-python 4.14.0.94
Uninstalling opencv-contrib-python-4.14.0.94:
  Successfully uninstalled opencv-contrib-python-4.14.0.94


In [ ]:
!pip install -q --no-cache-dir opencv-python-headless==4.10.0.84

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 220.8 MB/s eta 0:00:00


In [ ]:
%cd /content/deep-fake-shield

!python scripts/train_baseline.py \
    --train-csv artifacts/splits/train.csv \
    --val-csv artifacts/splits/val.csv \
    --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
    --pretrained \
    --epochs 1 \
    --limit 100 \
    --frames-per-video 2 \
    --batch-size 8 \
    --workers 2 \
    --output-dir artifacts/experiments/colab_sanity_001

/content/deep-fake-shield
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 154MB/s]
Traceback (most recent call last):
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 113, in <module>
    raise SystemExit(main())
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 96, in main
    val_metrics = run_epoch(model, val_loader, device)
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 30, in run_epoch
    for frames, targets in loader:
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1524, in _next_data
    idx, data = self._get_data()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1483, in 

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
!nvidia-smi

Tue Sep 15 13:18:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             30W /   70W |     183MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
Selected device: cuda
GPU: Tesla T4


In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/DeepFake-Shield/datasets/raw")

for folder in ["Celeb-Youtube-fake", "Celeb-real", "YouTube-real"]:
    videos = list((DATA_ROOT / folder).glob("*.mp4"))
    print(folder, len(videos))

    if videos:
        print("Example:", videos[0])

Celeb-Youtube-fake 5299
Example: /content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-Youtube-fake/4866.mp4
Celeb-real 482
Example: /content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-real/1.mp4
YouTube-real 230
Example: /content/drive/MyDrive/DeepFake-Shield/datasets/raw/YouTube-real/0.mp4


In [ ]:
import cv2

video_path = list((DATA_ROOT / "Celeb-real").glob("*.mp4"))[0]

cap = cv2.VideoCapture(str(video_path))

print("Video:", video_path)
print("Opened:", cap.isOpened())
print("Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))

ret, frame = cap.read()

print("First frame read:", ret)
print("Frame shape:", frame.shape if ret else None)

cap.release()

Video: /content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-real/1.mp4
Opened: True
Frame count: 350
FPS: 30.0
First frame read: True
Frame shape: (500, 942, 3)


In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless

Found existing installation: opencv-python-headless 4.10.0.84
Uninstalling opencv-python-headless-4.10.0.84:
  Successfully uninstalled opencv-python-headless-4.10.0.84


In [ ]:
!pip list | grep -i opencv

In [ ]:
!pip install --no-cache-dir --force-reinstall opencv-contrib-python-headless==4.11.0.86

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 MB 245.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 242.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, which is not installed.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, which is not installed.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.


In [ ]:
import sys

sys.path.insert(0, "/content/deep-fake-shield")

from ml.data.video_dataset import FaceCropper

cropper = FaceCropper()

print("FaceCropper initialized successfully.")
print("Detector empty:", cropper.detector.empty())

FaceCropper initialized successfully.
Detector empty: False


### What we want

The script should:

```text
Load pretrained model
        ↓
Load videos
        ↓
Extract frames
        ↓
Train
        ↓
Validate
        ↓
Save checkpoint
```

---

# CELL 22 — Check checkpoint



In [ ]:
from pathlib import Path

# Repository
REPO_ROOT = Path("/content/deep-fake-shield")

# Dataset on Google Drive
DATA_ROOT = Path("/content/drive/MyDrive/DeepFake-Shield/datasets/raw")

# Existing dataset splits
SPLITS_ROOT = REPO_ROOT / "artifacts" / "splits"

TRAIN_CSV = SPLITS_ROOT / "train.csv"
VAL_CSV = SPLITS_ROOT / "val.csv"
TEST_CSV = SPLITS_ROOT / "test.csv"

print("Repository exists:", REPO_ROOT.exists())
print("Dataset exists:", DATA_ROOT.exists())
print("Splits exists:", SPLITS_ROOT.exists())
print("Train CSV:", TRAIN_CSV.exists())
print("Val CSV:", VAL_CSV.exists())
print("Test CSV:", TEST_CSV.exists())

Repository exists: True
Dataset exists: True
Splits exists: True
Train CSV: True
Val CSV: True
Test CSV: True


In [ ]:
from pathlib import Path

SANITY_DIR = (
    REPO_ROOT
    / "artifacts"
    / "experiments"
    / "colab_sanity_001"
)

print("Experiment exists:", SANITY_DIR.exists())

if SANITY_DIR.exists():

    print("\nFiles:")

    for item in SANITY_DIR.iterdir():
        print(
            "📄",
            item.name
        )

Experiment exists: True

Files:


Look for:

```text
best_model.pt
```

---

# CELL 23 — Evaluate sanity model



In [ ]:
%cd /content/deep-fake-shield

/content/deep-fake-shield


In [ ]:
%cd /content/deep-fake-shield

!find artifacts -type f \( -name "*.pt" -o -name "*.pth" -o -name "*.ckpt" \) -print

/content/deep-fake-shield


In [ ]:
!find artifacts/experiments -maxdepth 2 -type d -print

artifacts/experiments
artifacts/experiments/colab_sanity_001
artifacts/experiments/submission_v1


In [ ]:
%cd /content/deep-fake-shield
!find artifacts -type f \( -name "*.pt" -o -name "*.pth" -o -name "*.ckpt" \) -print

/content/deep-fake-shield


In [ ]:
import torch
import cv2

print("GPU:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("OpenCV:", cv2.__version__)
print("CascadeClassifier:", hasattr(cv2, "CascadeClassifier"))

GPU: True
Device: Tesla T4
OpenCV: 4.11.0
CascadeClassifier: True


In [ ]:
%cd /content/deep-fake-shield

!python scripts/train_baseline.py \
    --train-csv artifacts/splits/train.csv \
    --val-csv artifacts/splits/val.csv \
    --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
    --pretrained \
    --epochs 1 \
    --limit 100 \
    --frames-per-video 2 \
    --batch-size 8 \
    --workers 2 \
    --output-dir artifacts/experiments/colab_sanity_001

/content/deep-fake-shield
{
  "epoch": 1,
  "train": {
    "accuracy": 0.73,
    "precision": 0.9259259259259259,
    "recall": 0.5,
    "f1": 0.6493506493506493,
    "roc_auc": 0.7792000000000001,
    "pr_auc": 0.8317532453854263,
    "false_positive_rate": 0.04,
    "false_negative_rate": 0.5,
    "support": 100,
    "loss": 0.6282301926612854
  },
  "validation": {
    "accuracy": 0.56,
    "precision": 0.7142857142857143,
    "recall": 0.2,
    "f1": 0.3125,
    "roc_auc": 0.598,
    "pr_auc": 0.6413366125280481,
    "false_positive_rate": 0.08,
    "false_negative_rate": 0.8,
    "support": 100,
    "loss": 0.693071894645691
  }
}
device=cuda output=artifacts/experiments/colab_sanity_001


In [ ]:
if SANITY_DIR.exists():

    !python scripts/evaluate_baseline.py \
        --csv artifacts/splits/test.csv \
        --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
        --checkpoint artifacts/experiments/colab_sanity_001/best_model.pt \
        --frames 2 \
        --limit 100 \
        --output-dir artifacts/evaluation/colab_sanity_001

{
  "device": "cuda",
  "videos": 100,
  "metrics": {
    "accuracy": 0.57,
    "precision": 0.7058823529411765,
    "recall": 0.24,
    "f1": 0.3582089552238806,
    "roc_auc": 0.6316,
    "pr_auc": 0.6444281831306727,
    "false_positive_rate": 0.1,
    "false_negative_rate": 0.76,
    "support": 100
  }
}


---

# CELL 24 — Read sanity metrics



In [ ]:
import json

metrics_file = (
    REPO_ROOT
    / "artifacts"
    / "evaluation"
    / "colab_sanity_001"
    / "test_metrics.json"
)

if metrics_file.exists():

    metrics = json.loads(
        metrics_file.read_text()
    )

    print(
        json.dumps(
            metrics,
            indent=2
        )
    )

else:
    print(
        "Metrics file not found:",
        metrics_file
    )

{
  "accuracy": 0.57,
  "precision": 0.7058823529411765,
  "recall": 0.24,
  "f1": 0.3582089552238806,
  "roc_auc": 0.6316,
  "pr_auc": 0.6444281831306727,
  "false_positive_rate": 0.1,
  "false_negative_rate": 0.76,
  "support": 100
}


This sanity run is only to prove the pipeline works.

**Don't worry if the metrics are poor on 100 videos.**

---

# 🚀 CELL 25 — FULL TRAINING

If Cell 21 completed successfully and produced a checkpoint, now we do the real training.



In [ ]:
if SANITY_DIR.exists() and (SANITY_DIR / "best_model.pt").exists():
    %cd /content/deep-fake-shield

    !python scripts/train_baseline.py \
        --train-csv artifacts/splits/train.csv \
        --val-csv artifacts/splits/val.csv \
        --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
        --pretrained \
        --epochs 5 \
        --limit 0 \
        --frames-per-video 4 \
        --batch-size 8 \
        --workers 2 \
        --output-dir artifacts/experiments/submission_v1
else:
    print("Sanity training not completed or best_model.pt not found. Please check Cell 21.")

/content/deep-fake-shield
Traceback (most recent call last):
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 113, in <module>
    raise SystemExit(main())
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 95, in main
    train_metrics = run_epoch(model, train_loader, device, optimizer)
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 30, in run_epoch
    for frames, targets in loader:
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1524, in _next_data
    idx, data = self._get_data()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1483, in _get_data
    success, data = self._try_get_data()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1310, in _try_get_data
    data = self._data_queue.get(

In [ ]:
%cd /content/deep-fake-shield

!find artifacts/experiments/submission_v1 -maxdepth 2 -type f -print 2>/dev/null

/content/deep-fake-shield


In [ ]:
%cd /content/deep-fake-shield

!python scripts/train_baseline.py \
    --train-csv artifacts/splits/train.csv \
    --val-csv artifacts/splits/val.csv \
    --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
    --pretrained \
    --epochs 2 \
    --limit 1000 \
    --frames-per-video 2 \
    --batch-size 16 \
    --workers 2 \
    --output-dir artifacts/experiments/submission_fast

/content/deep-fake-shield
Traceback (most recent call last):
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 113, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 95, in main
    train_metrics = run_epoch(model, train_loader, device, optimizer)
  File "/content/deep-fake-shield/scripts/train_baseline.py", line 34, in run_epoch
    logits = model(frames.reshape(batch * steps, channels, height, width))
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/torchvision/models/efficientnet.py", line 344, in forward
    return self._forward_impl(x)
          

In [ ]:
!nvidia-smi

Tue Sep 15 13:41:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             30W /   70W |       3MiB /  15360MiB |     16%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content/deep-fake-shield

!find artifacts/experiments/submission_v1 -maxdepth 2 -type f -print 2>/dev/null

/content/deep-fake-shield


In [ ]:
!find artifacts/experiments -maxdepth 2 -type f \( -name "*.pt" -o -name "*.json" -o -name "*.csv" \) -print

artifacts/experiments/colab_sanity_001/last_model.pt
artifacts/experiments/colab_sanity_001/best_model.pt
artifacts/experiments/colab_sanity_001/config.json
artifacts/experiments/colab_sanity_001/training_history.json


In [ ]:
import json
from pathlib import Path

history_file = Path(
    "/content/deep-fake-shield/artifacts/experiments/colab_sanity_001/training_history.json"
)

with open(history_file) as f:
    history = json.load(f)

print(json.dumps(history, indent=2))

[
  {
    "epoch": 1,
    "train": {
      "accuracy": 0.73,
      "precision": 0.9259259259259259,
      "recall": 0.5,
      "f1": 0.6493506493506493,
      "roc_auc": 0.7792000000000001,
      "pr_auc": 0.8317532453854263,
      "false_positive_rate": 0.04,
      "false_negative_rate": 0.5,
      "support": 100,
      "loss": 0.6282301926612854
    },
    "validation": {
      "accuracy": 0.56,
      "precision": 0.7142857142857143,
      "recall": 0.2,
      "f1": 0.3125,
      "roc_auc": 0.598,
      "pr_auc": 0.6413366125280481,
      "false_positive_rate": 0.08,
      "false_negative_rate": 0.8,
      "support": 100,
      "loss": 0.693071894645691
    }
  }
]


In [ ]:
config_file = Path(
    "/content/deep-fake-shield/artifacts/experiments/colab_sanity_001/config.json"
)

with open(config_file) as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "train_csv": "artifacts/splits/train.csv",
  "val_csv": "artifacts/splits/val.csv",
  "video_root": "/content/drive/MyDrive/DeepFake-Shield/datasets/raw",
  "epochs": 1,
  "limit": 100,
  "frames_per_video": 2,
  "batch_size": 8,
  "workers": 2,
  "output_dir": "artifacts/experiments/colab_sanity_001",
  "pretrained": true,
  "seed": 42
}


In [ ]:
!ls -lh artifacts/experiments/colab_sanity_001/*.pt

-rw-r--r-- 1 root root 16M Sep 15 13:25 artifacts/experiments/colab_sanity_001/best_model.pt
-rw-r--r-- 1 root root 16M Sep 15 13:25 artifacts/experiments/colab_sanity_001/last_model.pt


In [ ]:
%cd /content/deep-fake-shield

!python scripts/evaluate_baseline.py \
    --csv artifacts/splits/test.csv \
    --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
    --checkpoint artifacts/experiments/colab_sanity_001/best_model.pt \
    --frames 2 \
    --limit 100 \
    --output-dir artifacts/evaluation/colab_sanity_001

/content/deep-fake-shield
{
  "device": "cuda",
  "videos": 100,
  "metrics": {
    "accuracy": 0.57,
    "precision": 0.7058823529411765,
    "recall": 0.24,
    "f1": 0.3582089552238806,
    "roc_auc": 0.6316,
    "pr_auc": 0.6444281831306727,
    "false_positive_rate": 0.1,
    "false_negative_rate": 0.76,
    "support": 100
  }
}


In [ ]:
!nvidia-smi

Tue Sep 15 13:44:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
import cv2

print("PyTorch CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("OpenCV:", cv2.__version__)
print("CascadeClassifier:", hasattr(cv2, "CascadeClassifier"))

PyTorch CUDA: True
GPU: Tesla T4
OpenCV: 4.11.0
CascadeClassifier: True


In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/DeepFake-Shield/datasets/raw")

videos = list((DATA_ROOT / "Celeb-real").glob("*.mp4"))

print("Videos found:", len(videos))
print("Testing:", videos[0])

Videos found: 482
Testing: /content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-real/1.mp4


In [ ]:
from pathlib import Path
import sys
import time

sys.path.insert(0, "/content/deep-fake-shield")

from ml.data.video_dataset import VideoFrameDataset

REPO_ROOT = Path("/content/deep-fake-shield")
DATA_ROOT = Path("/content/drive/MyDrive/DeepFake-Shield/datasets/raw")
TRAIN_CSV = REPO_ROOT / "artifacts/splits/train.csv"

start = time.time()

dataset = VideoFrameDataset(
    TRAIN_CSV,                 # Path object, NOT string
    frames_per_video=2,
    transform=None,
    video_root=DATA_ROOT
)

print("Dataset initialized in:", time.time() - start, "seconds")
print("Dataset length:", len(dataset))

Dataset initialized in: 0.03896594047546387 seconds
Dataset length: 4207


In [ ]:
%cd /content/deep-fake-shield

!sed -n '1,150p' ml/data/video_dataset.py

/content/deep-fake-shield
"""Lazy video-frame dataset with deterministic sampling and face crops."""

from __future__ import annotations

import csv
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset


def sample_frame_indices(frame_count: int, count: int) -> np.ndarray:
    if frame_count <= 0:
        return np.zeros(count, dtype=np.int64)
    return np.linspace(0, max(frame_count - 1, 0), count, dtype=np.int64)


class FaceCropper:
    def __init__(self, margin: float = 0.2) -> None:
        self.margin = margin
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        self.detector = cv2.CascadeClassifier(cascade_path)

    def crop(self, frame: np.ndarray) -> np.ndarray | None:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = self.detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(32, 32))
        if len

In [ ]:
from pathlib import Path

file = Path("/content/deep-fake-shield/ml/data/video_dataset.py")

text = file.read_text()

old = '''        if self.transform is not None:
            tensors = [self.transform(frame) for frame in frames]
        else:
            tensors = [torch.from_numpy(np.array(frame, copy=True)).permute(2, 0, 1).float() / 255 for frame in frames]
        tensors = [
            torch.from_numpy(np.array(frame, copy=True)).permute(2, 0, 1).float() / 255
            if not isinstance(frame, torch.Tensor)
            else frame
            for frame in tensors
        ]
'''

new = '''        # Ensure every frame has the same spatial dimensions.
        # EfficientNet-B0 expects a consistent image size.
        if self.transform is not None:
            tensors = [self.transform(frame) for frame in frames]
        else:
            tensors = []
            for frame in frames:
                frame = frame.resize((224, 224), Image.Resampling.BILINEAR)
                array = np.array(frame, copy=True)
                tensor = torch.from_numpy(array).permute(2, 0, 1).float() / 255.0
                tensors.append(tensor)

        # Make sure transformed tensors also have consistent dimensions.
        normalized_tensors = []
        for tensor in tensors:
            if not isinstance(tensor, torch.Tensor):
                tensor = torch.from_numpy(
                    np.array(tensor, copy=True)
                ).permute(2, 0, 1).float() / 255.0

            if tensor.ndim == 3 and tensor.shape[-2:] != (224, 224):
                tensor = torch.nn.functional.interpolate(
                    tensor.unsqueeze(0),
                    size=(224, 224),
                    mode="bilinear",
                    align_corners=False,
                ).squeeze(0)

            normalized_tensors.append(tensor)

        tensors = normalized_tensors
'''

if old not in text:
    print("ERROR: Expected code block was not found.")
else:
    file.write_text(text.replace(old, new))
    print("video_dataset.py updated successfully.")

video_dataset.py updated successfully.


In [ ]:
%cd /content/deep-fake-shield

!nl -ba ml/data/video_dataset.py | sed -n '80,130p'

/content/deep-fake-shield
    80	            image = crop if crop is not None else frame
    81	            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    82	            frames.append(Image.fromarray(image))
    83	        capture.release()
    84	        if not frames:
    85	            frames = [Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))]
    86	        # Ensure every frame has the same spatial dimensions.
    87	        # EfficientNet-B0 expects a consistent image size.
    88	        if self.transform is not None:
    89	            tensors = [self.transform(frame) for frame in frames]
    90	        else:
    91	            tensors = []
    92	            for frame in frames:
    93	                frame = frame.resize((224, 224), Image.Resampling.BILINEAR)
    94	                array = np.array(frame, copy=True)
    95	                tensor = torch.from_numpy(array).permute(2, 0, 1).float() / 255.0
    96	                tensors.append(tensor)
    97	
    98	

In [ ]:
%cd /content/deep-fake-shield

!nl -ba ml/data/video_dataset.py | sed -n '1,130p'

/content/deep-fake-shield
     1	"""Lazy video-frame dataset with deterministic sampling and face crops."""
     2	
     3	from __future__ import annotations
     4	
     5	import csv
     6	from pathlib import Path
     7	from typing import Any
     8	
     9	import cv2
    10	import numpy as np
    11	import torch
    12	from PIL import Image
    13	from torch.utils.data import Dataset
    14	
    15	
    16	def sample_frame_indices(frame_count: int, count: int) -> np.ndarray:
    17	    if frame_count <= 0:
    18	        return np.zeros(count, dtype=np.int64)
    19	    return np.linspace(0, max(frame_count - 1, 0), count, dtype=np.int64)
    20	
    21	
    22	class FaceCropper:
    23	    def __init__(self, margin: float = 0.2) -> None:
    24	        self.margin = margin
    25	        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    26	        self.detector = cv2.CascadeClassifier(cascade_path)
    27	
    28	    def crop(self, frame: np.ndarray)

In [ ]:
from pathlib import Path

file = Path("/content/deep-fake-shield/ml/data/video_dataset.py")

text = file.read_text()

start = text.index("    def __getitem__(self, index: int)")

new_method = '''    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        record = self.records[index]

        video_path = self.resolve_video_path(record["video_path"])
        capture = cv2.VideoCapture(str(video_path))

        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

        frames = []

        for frame_index in sample_frame_indices(
            frame_count, self.frames_per_video
        ):
            capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ok, frame = capture.read()

            if not ok:
                continue

            crop = self.cropper.crop(frame)
            image = crop if crop is not None else frame

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Force every frame to the same size.
            image = cv2.resize(
                image,
                (224, 224),
                interpolation=cv2.INTER_LINEAR,
            )

            frames.append(Image.fromarray(image))

        capture.release()

        if not frames:
            frames = [
                Image.fromarray(
                    np.zeros((224, 224, 3), dtype=np.uint8)
                )
            ]

        tensors = []

        for frame in frames:

            # Always enforce 224x224 before transform.
            frame = frame.resize(
                (224, 224),
                Image.Resampling.BILINEAR,
            )

            if self.transform is not None:
                tensor = self.transform(frame)
            else:
                array = np.array(frame, copy=True)
                tensor = (
                    torch.from_numpy(array)
                    .permute(2, 0, 1)
                    .float()
                    / 255.0
                )

            # Convert non-tensor output if necessary.
            if not isinstance(tensor, torch.Tensor):
                tensor = torch.from_numpy(
                    np.array(tensor, copy=True)
                )

                if tensor.ndim == 3 and tensor.shape[-1] in (1, 3):
                    tensor = tensor.permute(2, 0, 1)

                tensor = tensor.float() / 255.0

            # Final safety resize.
            if tensor.ndim == 3 and tensor.shape[-2:] != (224, 224):
                tensor = torch.nn.functional.interpolate(
                    tensor.unsqueeze(0),
                    size=(224, 224),
                    mode="bilinear",
                    align_corners=False,
                ).squeeze(0)

            tensors.append(tensor)

        # Duplicate the final frame if necessary.
        while len(tensors) < self.frames_per_video:
            tensors.append(tensors[-1].clone())

        return (
            torch.stack(tensors[: self.frames_per_video]),
            torch.tensor(
                int(record["label"]),
                dtype=torch.long,
            ),
        )
'''

file.write_text(text[:start] + new_method + "\n")

print("✅ video_dataset.py has been fixed.")

✅ video_dataset.py has been fixed.


In [ ]:
from pathlib import Path
import sys
import time

REPO_ROOT = Path("/content/deep-fake-shield")
DATA_ROOT = Path("/content/drive/MyDrive/DeepFake-Shield/datasets/raw")
TRAIN_CSV = REPO_ROOT / "artifacts/splits/train.csv"

sys.path.insert(0, str(REPO_ROOT))

from ml.data.video_dataset import VideoFrameDataset

dataset = VideoFrameDataset(
    TRAIN_CSV,
    frames_per_video=2,
    transform=None,
    video_root=DATA_ROOT
)

print("Dataset length:", len(dataset))

start = time.time()

sample = dataset[0]

elapsed = time.time() - start

print("Loaded in:", round(elapsed, 2), "seconds")
print("Frames shape:", sample[0].shape)
print("Label:", sample[1])

Dataset length: 4207
Loaded in: 0.6 seconds
Frames shape: torch.Size([2, 3, 224, 224])
Label: tensor(1)


In [ ]:
start = time.time()

sample = dataset[0]

print("First sample loaded in:", time.time() - start, "seconds")
print("Sample type:", type(sample))

if isinstance(sample, tuple):
    print("Number of returned items:", len(sample))
    for i, item in enumerate(sample):
        print(i, type(item), getattr(item, "shape", None))

First sample loaded in: 0.292527437210083 seconds
Sample type: <class 'tuple'>
Number of returned items: 2
0 <class 'torch.Tensor'> torch.Size([2, 3, 224, 224])
1 <class 'torch.Tensor'> torch.Size([])


In [ ]:
from ml.data.video_dataset import VideoFrameDataset
import time

dataset = VideoFrameDataset(
    TRAIN_CSV,
    frames_per_video=2,
    transform=None,
    video_root=DATA_ROOT
)

print("Dataset length:", len(dataset))

start = time.time()

sample = dataset[0]

print("Time:", round(time.time() - start, 2), "seconds")
print("Shape:", sample[0].shape)
print("Label:", sample[1])

Dataset length: 4207
Time: 0.98 seconds
Shape: torch.Size([2, 3, 224, 224])
Label: tensor(1)


In [ ]:
%cd /content/deep-fake-shield

!python scripts/train_baseline.py \
    --train-csv artifacts/splits/train.csv \
    --val-csv artifacts/splits/val.csv \
    --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
    --pretrained \
    --epochs 2 \
    --limit 1000 \
    --frames-per-video 2 \
    --batch-size 16 \
    --workers 2 \
    --output-dir artifacts/experiments/submission_final

/content/deep-fake-shield
{
  "epoch": 1,
  "train": {
    "accuracy": 0.7855711422845691,
    "precision": 0.7930327868852459,
    "recall": 0.774,
    "f1": 0.7834008097165992,
    "roc_auc": 0.8667710843373494,
    "pr_auc": 0.8700386414362444,
    "false_positive_rate": 0.20281124497991967,
    "false_negative_rate": 0.226,
    "support": 998,
    "loss": 0.5058757006405351
  },
  "validation": {
    "accuracy": 0.541019955654102,
    "precision": 0.9635036496350365,
    "recall": 0.4981132075471698,
    "f1": 0.6567164179104478,
    "roc_auc": 0.7920648915535179,
    "pr_auc": 0.9589464739232781,
    "false_positive_rate": 0.14018691588785046,
    "false_negative_rate": 0.5018867924528302,
    "support": 902,
    "loss": 0.7935367470768233
  }
}
{
  "epoch": 2,
  "train": {
    "accuracy": 0.9218436873747495,
    "precision": 0.9359504132231405,
    "recall": 0.906,
    "f1": 0.9207317073170732,
    "roc_auc": 0.971710843373494,
    "pr_auc": 0.9703615420662247,
    "false_positiv

In [ ]:
%cd /content/deep-fake-shield

!python scripts/evaluate_baseline.py \
    --csv artifacts/splits/test.csv \
    --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
    --checkpoint artifacts/experiments/submission_final/best_model.pt \
    --frames 2 \
    --limit 902 \
    --output-dir artifacts/evaluation/submission_final

/content/deep-fake-shield
{
  "device": "cuda",
  "videos": 902,
  "metrics": {
    "accuracy": 0.5543237250554324,
    "precision": 0.9734939759036144,
    "recall": 0.5081761006289308,
    "f1": 0.6677685950413224,
    "roc_auc": 0.830094633515547,
    "pr_auc": 0.9679018227235412,
    "false_positive_rate": 0.102803738317757,
    "false_negative_rate": 0.4918238993710692,
    "support": 902
  }
}


### This is your main training run.

It uses:

```text
4,207 training videos
902 validation videos
4 frames/video
5 epochs
GPU
```

The test set is **not used during training**.

---

# Important: Watch the output

You should see training progress similar to:

```text
Epoch 1/5
Training ...
Validation ...

Epoch 2/5
Training ...
Validation ...

...
```

and eventually:

```text
best_model.pt
```

### Don't close the Colab tab while this is running.

Also, don't run another training cell simultaneously.

---

# CELL 26 — Check the trained model

After training completes:



In [ ]:
from pathlib import Path

FULL_DIR = (
    REPO_ROOT
    / "artifacts"
    / "experiments"
    / "submission_v1"
)

print("Training directory:", FULL_DIR)

for item in sorted(FULL_DIR.iterdir()):
    print(
        item.name,
        "→",
        round(item.stat().st_size / 1024**2, 2),
        "MB"
        if item.is_file()
        else ""
    )

Training directory: /content/deep-fake-shield/artifacts/experiments/submission_v1


Find:

```text
best_model.pt
```

---

# CELL 27 — FINAL TEST

Now evaluate against your **untouched 902-video test set**.



In [ ]:
if (FULL_DIR / "best_model.pt").exists():
    !python scripts/evaluate_baseline.py \
        --csv artifacts/splits/test.csv \
        --video-root "/content/drive/MyDrive/DeepFake-Shield/datasets/raw" \
        --checkpoint artifacts/experiments/submission_v1/best_model.pt \
        --frames 4 \
        --limit 0 \
        --output-dir artifacts/evaluation/submission_v1
else:
    print("Full training not completed or best_model.pt not found. Cannot perform final test.")

Full training not completed or best_model.pt not found. Cannot perform final test.


This is your final evaluation.

---

# CELL 28 — Display final metrics



In [ ]:
import json
from pathlib import Path

FINAL_METRICS = (
    REPO_ROOT
    / "artifacts"
    / "evaluation"
    / "submission_v1"
    / "test_metrics.json"
)

if not FINAL_METRICS.exists():

    print(
        "❌ Final metrics file not found:"
    )

    print(FINAL_METRICS)

else:

    metrics = json.loads(
        FINAL_METRICS.read_text()
    )

    print(
        json.dumps(
            metrics,
            indent=2
        )
    )

❌ Final metrics file not found:
/content/deep-fake-shield/artifacts/evaluation/submission_v1/test_metrics.json


### Save this output.

These are the numbers you'll want for your project presentation/submission.

---

# CELL 29 — Test a real video



In [ ]:
real_videos = []

for folder_name in [
    "Celeb-real",
    "YouTube-real"
]:

    folder = DATA_ROOT / folder_name

    if folder.exists():

        real_videos.extend(
            folder.glob("*.mp4")
        )

print("Real videos found:", len(real_videos))

if real_videos:
    print("Sample:", real_videos[0])

Real videos found: 712
Sample: /content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-real/1.mp4


---

# CELL 30 — Test a fake video



In [ ]:
fake_videos = list(
    (DATA_ROOT / "Celeb-Youtube-fake").glob("*.mp4")
)

print("Fake videos found:", len(fake_videos))

if fake_videos:
    print("Sample:", fake_videos[0])

Fake videos found: 5299
Sample: /content/drive/MyDrive/DeepFake-Shield/datasets/raw/Celeb-Youtube-fake/4866.mp4


---

# CELL 31 — Run inference on a fake video



In [ ]:
if fake_videos and (FULL_DIR / "best_model.pt").exists():
    fake_video = fake_videos[0]

    print("Testing:")
    print(fake_video)

    !python scripts/infer_video.py \
        --video "{fake_video}" \
        --checkpoint artifacts/experiments/submission_v1/best_model.pt \
        --frames 4
else:
    print("Fake video list is empty or full training model not found. Cannot run inference.")

Fake video list is empty or full training model not found. Cannot run inference.


---

# CELL 32 — Run inference on a real video



In [ ]:
if real_videos and (FULL_DIR / "best_model.pt").exists():
    real_video = real_videos[0]

    print("Testing:")
    print(real_video)

    !python scripts/infer_video.py \
        --video "{real_video}" \
        --checkpoint artifacts/experiments/submission_v1/best_model.pt \
        --frames 4
else:
    print("Real video list is empty or full training model not found. Cannot run inference.")

Real video list is empty or full training model not found. Cannot run inference.


You want the model to distinguish the two correctly.

---

# CELL 33 — Copy production model to Google Drive



In [ ]:
from pathlib import Path
import shutil

# Paths
MODEL_SRC = Path(
    "/content/deep-fake-shield/artifacts/experiments/submission_final/best_model.pt"
)

DRIVE_MODEL_ROOT = Path(
    "/content/drive/MyDrive/DeepFake-Shield/models"
)

# Create Drive models directory
DRIVE_MODEL_ROOT.mkdir(parents=True, exist_ok=True)

# Production model name
PRODUCTION_MODEL = DRIVE_MODEL_ROOT / "deepfake_shield_video_v1.pt"

# Check source model exists
if not MODEL_SRC.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_SRC}")

# Copy model to Google Drive
shutil.copy2(MODEL_SRC, PRODUCTION_MODEL)

print("✅ Model copied successfully!")
print(f"📦 Model: {PRODUCTION_MODEL}")
print(f"📏 Size: {PRODUCTION_MODEL.stat().st_size / (1024**2):.2f} MB")

✅ Model copied successfully!
📦 Model: /content/drive/MyDrive/DeepFake-Shield/models/deepfake_shield_video_v1.pt
📏 Size: 15.58 MB


In [ ]:
from pathlib import Path

PRODUCTION_MODEL = Path(
    "/content/drive/MyDrive/DeepFake-Shield/models/deepfake_shield_video_v1.pt"
)

print("Exists:", PRODUCTION_MODEL.exists())
print("Path:", PRODUCTION_MODEL)
print("Size:", f"{PRODUCTION_MODEL.stat().st_size / (1024**2):.2f} MB")

Exists: True
Path: /content/drive/MyDrive/DeepFake-Shield/models/deepfake_shield_video_v1.pt
Size: 15.58 MB


In [ ]:
import shutil
from pathlib import Path

SOURCE_MODEL = (
    REPO_ROOT
    / "artifacts"
    / "experiments"
    / "submission_v1"
    / "best_model.pt"
)

PRODUCTION_MODEL = (
    DRIVE_MODEL_ROOT
    / "deepfake_shield_video_v1.pt"
)

if not SOURCE_MODEL.exists():
    raise FileNotFoundError(
        f"Model not found: {SOURCE_MODEL}"
    )

shutil.copy2(
    SOURCE_MODEL,
    PRODUCTION_MODEL
)

print("✅ Production model saved:")
print(PRODUCTION_MODEL)

print(
    "Size:",
    round(
        PRODUCTION_MODEL.stat().st_size / 1024**2,
        2
    ),
    "MB"
)

NameError: name 'DRIVE_MODEL_ROOT' is not defined

Now your model is permanently in:

```text
Google Drive
└── DeepFake-Shield/
    └── models/
        └── deepfake_shield_video_v1.pt
```

---

# CELL 34 — Download model to Mac



In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/DeepFake-Shield/models/deepfake_shield_video_v1.pt"
)

FileNotFoundError: Cannot find file: /content/drive/MyDrive/DeepFake-Shield/models/deepfake_shield_video_v1.pt

Put the downloaded file into your local project.

For example:

```text
deep-fake-shield/
│
├── backend/
│   ├── main.py
│   ├── inference.py
│   └── models/
│       └── deepfake_shield_video_v1.pt
│
└── frontend/
```

---

# CELL 35 — Save final metrics to Drive



In [ ]:
FINAL_REPORT_DIR = (
    Path("/content/drive/MyDrive/DeepFake-Shield")
    / "submission_results"
)

FINAL_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if FINAL_METRICS.exists():

    shutil.copy2(
        FINAL_METRICS,
        FINAL_REPORT_DIR / "test_metrics.json"
    )

    print(
        "✅ Metrics saved to:",
        FINAL_REPORT_DIR / "test_metrics.json"
    )

---

# Your final project state

After everything succeeds, you'll have:

```text
                    DeepFake Shield
                           │
             ┌─────────────┴─────────────┐
             │                           │
         FRONTEND                    BACKEND
          ✅ DONE                      ✅ DONE
                                         │
                                         ▼
                                  MODEL INFERENCE
                                         │
                                         ▼
                              deepfake_shield_video_v1.pt
                                         │
                                         ▼
                              REAL / FAKE + CONFIDENCE
```

And your ML results:

```text
Dataset
│
├── 5,299 fake
├──   712 real
└── 6,011 labeled
       │
       ▼
Train: 4,207
Val:     902
Test:    902
       │
       ▼
Pretrained Deep Learning Model
       │
       ▼
5 epochs
       │
       ▼
best_model.pt
       │
       ▼
902-video final test
       │
       ▼
Metrics
       │
       ▼
Production model
```